# Laborator 5 – Data Preparation, Feature Engineering & Feature Selection

Set de date: Students Performance in Exams (`students.csv`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Explorarea datelor
df = pd.read_csv('students.csv')
print("Primele 5 inregistrari:")
print(df.head())

In [ ]:
# Structura dataset-ului
print("\nStructura:")
print(df.info())
print("\nStatistici descriptive:")
print(df.describe())

In [ ]:
# Valori lipsa
print("\nValori lipsa per coloana:")
print(df.isnull().sum())

In [ ]:
# 2. Identificarea tipurilor de variabile
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f"Variabile categorice: {cat_cols}")
print(f"Variabile numerice:   {num_cols}")

In [ ]:
# 3. Curatarea datelor
# Inlocuire valori numerice lipsa cu mediana
for col in num_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

# Inlocuire valori categorice lipsa cu 'Unknown'
for col in cat_cols:
    df[col].fillna('Unknown', inplace=True)

print("Valori lipsa dupa curatare:")
print(df.isnull().sum())

In [ ]:
# 4. Encoding variabile categorice
# Label Encoding pentru variabila binara 'gender'
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])
print("Label Encoding 'gender':", dict(zip(le.classes_, le.transform(le.classes_))))

# One-Hot Encoding pentru celelalte variabile categorice
cat_to_encode = ['race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
df_encoded = pd.get_dummies(df, columns=cat_to_encode, drop_first=False)
print(f"\nColoane dupa One-Hot Encoding: {df_encoded.shape[1]}")
print(df_encoded.head(2).to_string())

In [ ]:
# 5. Feature Engineering
# average_score
df['average_score'] = df[['math score','reading score','writing score']].mean(axis=1).round(2)

# performance_level
def perf_level(score):
    if score < 50: return 'low'
    elif score <= 70: return 'medium'
    else: return 'high'

df['performance_level'] = df['average_score'].apply(perf_level)

# is_prepared (binar)
df['is_prepared'] = (df['test preparation course'] == 'completed').astype(int)

print(df[['average_score','performance_level','is_prepared']].head(10).to_string(index=False))
print("\nDistributie performance_level:")
print(df['performance_level'].value_counts())

In [ ]:
# 6. Feature Selection – justificare
print("Coloane disponibile:")
for col in df.columns:
    print(f"  {col}")

# Coloanele originale categorice si 'gender' sunt redundante dupa encoding
# Pastram coloanele numerice + is_prepared + average_score; eliminam textuale originale
cols_to_drop = ['gender', 'race/ethnicity', 'parental level of education',
                'lunch', 'test preparation course', 'performance_level']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df_clean = df.drop(columns=cols_to_drop)

# Verificam corelatie pentru redundanta
corr = df_clean[['math score','reading score','writing score','average_score']].corr()
print("\nMatrice de corelatie (scoruri):")
print(corr.round(2).to_string())
print("\n→ 'average_score' este corelat cu celelalte scoruri, dar il pastram ca feature inginerit.")

In [ ]:
# Vizualizare distributii
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ['math score','reading score','writing score']):
    df[col].hist(bins=20, ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.savefig('lab5_histograms.png', dpi=80)
plt.show()

In [ ]:
# 7. Scalarea datelor cu StandardScaler
num_features = ['math score', 'reading score', 'writing score', 'average_score']
scaler = StandardScaler()
scaled = scaler.fit_transform(df[num_features])
df_scaled = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in num_features])

print("Inainte de scalare:")
print(df[num_features].describe().round(2).to_string())
print("\nDupa scalare (StandardScaler):")
print(df_scaled.describe().round(2).to_string())
print("\nExplicatie: StandardScaler centreaza datele la media=0 si std=1.")
print("Algoritmii bazati pe distanta (KNN, K-Means) sunt sensibili la scara valorilor.")

In [ ]:
# 8. Pregatirea dataset-ului final X, y
X = df[num_features + ['is_prepared', 'gender_encoded']].copy()
X[num_features] = scaler.transform(X[num_features])
y = df['performance_level']

print(f"Dimensiuni X: {X.shape}")
print(f"Dimensiuni y: {y.shape}")
print(f"Clase: {y.unique()}")
print(X.head())

In [ ]:
# 9. Interpretare
print("=== Interpretare ===")
print()
print("1. Cele mai importante caracteristici: average_score, math score, writing score")
print("   => scorurile academice sunt cel mai direct corelate cu performanta.")
print()
print("2. Impactul scalarii: aduce toate variabilele numerice pe aceeasi scara [medie=0, std=1],")
print("   evitand ca variabilele cu valori mari sa domine modelele bazate pe distanta.")
print()
print("3. Fara feature selection ar exista redundanta intre math/reading/writing/average_score,")
print("   crescand complexitatea modelului si riscul de overfitting.")
